# Netflix Data Cleaning and Datetime Parsing

## Part 1 – Quantifying Missing Data

In [1]:
import pandas as pd

netflix_titles = pd.read_csv("netflix_titles.csv")
netflix_titles.shape

(8807, 12)

In [2]:
netflix_titles.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [3]:
missing_pct = (netflix_titles.isnull().sum() / len(netflix_titles) * 100).round(2)
missing_pct.sort_values(ascending=False)

director        29.91
country          9.44
cast             9.37
date_added       0.11
rating           0.05
duration         0.03
show_id          0.00
type             0.00
title            0.00
release_year     0.00
listed_in        0.00
description      0.00
dtype: float64

#### Part 1 Summary:

- The dataset has 8,807 rows and 12 columns.
- `director` has the worst data gap: 2,634 values are missing (29.91%), almost 1 in 3 titles.
- `country` (9.44%) and `cast` (9.37%) come next.
- `date_added`, `rating`, and `duration` are missing less than 0.2%, so their impact is very small.
- The other six columns have no missing values.

## Part 2 – Cleaning Strategies (Dropping vs. Imputing)

In [4]:
# Strategy B experiment copy (made BEFORE Strategy A changes anything)
netflix_dropped = netflix_titles.copy()

In [5]:
# Strategy A: fill missing directors with "Unknown"
netflix_titles["director"] = netflix_titles["director"].fillna("Unknown")

# Verify: missing count should now be 0
netflix_titles["director"].isnull().sum()

np.int64(0)

In [6]:
# Strategy B: drop rows missing director OR cast (on the copy)
netflix_dropped = netflix_dropped.dropna(subset=["director", "cast"])

netflix_dropped.shape

(5700, 12)

In [7]:
rows_before = len(netflix_titles)
rows_after = len(netflix_dropped)
rows_lost = rows_before - rows_after

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Rows lost:   {rows_lost} ({rows_lost / rows_before * 100:.2f}%)")

Rows before: 8807
Rows after:  5700
Rows lost:   3107 (35.28%)


In [8]:
print("Before:\n", netflix_titles["type"].value_counts())
print("\nAfter:\n", netflix_dropped["type"].value_counts())

Before:
 type
Movie      6131
TV Show    2676
Name: count, dtype: int64

After:
 type
Movie      5522
TV Show     178
Name: count, dtype: int64


#### Part 2 Summary:

**Strategy A (Imputation)** kept all 8,807 rows. The `director` missing count dropped from 2,634 to 0.

**Strategy B (Row Dropping)** left 5,700 rows.
**Rows lost:** 8,807 − 5,700 = **3,107 rows (35.28%)**.

The loss was not spread evenly across content types:

| Type | Before | After | Lost |
|---|---|---|---|
| Movie | 6,131 | 5,522 | 9.93% |
| TV Show | 2,676 | 178 | 93.35% |

**Scenario:** Imagine an analyst wants to compare Movies and TV Shows,
for example, "Does Netflix add more TV Shows than Movies over time?"
After Strategy B, TV Shows fall from about 30% of the catalog to only 3%.
The analysis would wrongly conclude that Netflix barely offers TV Shows.
This happens because TV Shows often have no single director, so dropping
rows with a missing `director` removes almost all of them.
In this case, filling with "Unknown" (Strategy A) is the safer choice.

## Part 3 – Processing Datetime Data

In [9]:
netflix_titles.dtypes

show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

In [10]:
# Remove extra spaces, then convert text to datetime
netflix_titles["date_added"] = pd.to_datetime(netflix_titles["date_added"].str.strip())

netflix_titles.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   str           
 1   type          8807 non-null   str           
 2   title         8807 non-null   str           
 3   director      8807 non-null   str           
 4   cast          7982 non-null   str           
 5   country       7976 non-null   str           
 6   date_added    8797 non-null   datetime64[us]
 7   release_year  8807 non-null   int64         
 8   rating        8803 non-null   str           
 9   duration      8804 non-null   str           
 10  listed_in     8807 non-null   str           
 11  description   8807 non-null   str           
dtypes: datetime64[us](1), int64(1), str(10)
memory usage: 825.8 KB


In [11]:
netflix_titles["day_added"] = netflix_titles["date_added"].dt.day

netflix_titles.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,day_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",25.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",24.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,24.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,NaN,NaN,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",24.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,24.0


#### Part 3 Summary:

`.dtypes` showed that `date_added` was stored as text (e.g. "September 25, 2021").
The instructions expect `object`, but my output shows `str` because I'm using Pandas 3.0,
which gives text columns a dedicated string dtype. Either way, it is plain text, not a date.

I removed extra spaces with `.str.strip()` and converted the column with `pd.to_datetime()`.
`.info()` confirms it is now `datetime64[us]` (Pandas 3.0 uses microseconds by default).
The 10 missing dates became `NaT`.

Using the `.dt` accessor, I extracted the day of the month into a new column, `day_added`.
It shows as a float (e.g. 25.0) because the column contains `NaT` values.